# 🎓 LangChain Student Academic Advisor Agent (Google Gemini & SQLite)

## Problem Statement
You are given a SQLite database containing student information and marks. Build a **LangChain Agent** using **Google Gemini** that can answer student-related questions by dynamically selecting and executing the appropriate tools.

### Database Table: `students`
| student_id | name | department | python | database | ai | web |
| :--- | :--- | :--- | :---: | :---: | :---: | :---: |
| 22CS045 | Dhanushya | Computer Science | 85 | 72 | 90 | 78 |
| 22CS046 | Rahul | Computer Science | 65 | 70 | 68 | 72 |
| 22CS047 | Priya | Information Technology | 92 | 88 | 95 | 90 |
| 22CS048 | Arun | Information Technology | 55 | 60 | 58 | 62 |
| 22CS049 | Meena | Computer Science | 78 | 85 | 80 | 88 |

### Required Tools:
1. `get_student_info(student_id)`: Returns student name and department.
2. `get_student_marks(student_id)`: Returns python, database, ai, and web marks.
3. `calculator(expression)`: Safely computes total marks and average marks.
4. `get_passing_rules()`: Returns university rules (min subject mark: 35, min overall average: 40%).

### Agent Requirement:
The agent must **autonomously** determine which tools are required based on the user's question, chaining tools as needed without any hardcoded logic.

In [1]:
# Step 1: Install required packages
!pip install -q -U langchain langchain-community langchain-core langchain-google-genai python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install -q -U langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Hope-elite-agentic-course\sodak-day-5\.venv\Scripts\python.exe -m pip install --upgrade pip


In [3]:
# Step 2: Configure Google Gemini API Key & Optimize Windows Network
import os
import socket
from dotenv import load_dotenv

# Fix Windows IPv6 20-second timeout by prioritizing IPv4
_orig_getaddrinfo = socket.getaddrinfo
def _getaddrinfo_ipv4(*args, **kwargs):
    res = _orig_getaddrinfo(*args, **kwargs)
    ipv4 = [r for r in res if r[0] == socket.AF_INET]
    return ipv4 if ipv4 else res
socket.getaddrinfo = _getaddrinfo_ipv4

# Load key from .env or Colab Secrets
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass

if not api_key:
    api_key = "your_gemini_api_key_here"

os.environ["GEMINI_API_KEY"] = api_key
os.environ["GOOGLE_API_KEY"] = api_key
print("Google Gemini API Key configured and IPv4 network speed optimized!")


Google Gemini API Key configured successfully!


In [4]:
# Step 3: Initialize SQLite Database 'students.db'
import sqlite3

conn = sqlite3.connect('students.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS students (
    student_id TEXT PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT NOT NULL,
    python INTEGER NOT NULL,
    database INTEGER NOT NULL,
    ai INTEGER NOT NULL,
    web INTEGER NOT NULL
)
''')

students_data = [
    ('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78),
    ('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72),
    ('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90),
    ('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62),
    ('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88),
]

cursor.executemany('''
INSERT OR REPLACE INTO students (student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
''', students_data)

conn.commit()
conn.close()
print('Database students.db initialized with 5 student records.')

Database students.db initialized with 5 student records.


In [5]:
# Step 4: Define the 4 LangChain tools using @tool


import ast
import operator
from langchain_core.tools import tool

@tool
def get_student_info(student_id: str) -> dict:
    '''Returns student name and department given their student_id.'''
    conn = sqlite3.connect('students.db')
    cursor = conn.cursor()
    cursor.execute('SELECT name, department FROM students WHERE student_id = ?', (student_id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return {'student_id': student_id, 'name': row[0], 'department': row[1]}
    return {'error': f'Student with ID {student_id} not found.'}

@tool
def get_student_marks(student_id: str) -> dict:
    '''Returns python mark, database mark, ai mark, and web mark for a student.'''
    conn = sqlite3.connect('students.db')
    cursor = conn.cursor()
    cursor.execute('SELECT python, database, ai, web FROM students WHERE student_id = ?', (student_id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return {
            'student_id': student_id,
            'python': row[0],
            'database': row[1],
            'ai': row[2],
            'web': row[3],
        }
    return {'error': f'Student with ID {student_id} not found.'}

@tool
def calculator(expression: str) -> str:
    '''Used to calculate total marks and average marks from an arithmetic expression like "85 + 72 + 90 + 78" or "325 / 4".'''
    operators = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.USub: operator.neg,
    }
    def eval_expr(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        elif isinstance(node, ast.BinOp):
            return operators[type(node.op)](eval_expr(node.left), eval_expr(node.right))
        elif isinstance(node, ast.UnaryOp):
            return operators[type(node.op)](eval_expr(node.operand))
        raise ValueError(f'Unsupported operation: {node}')
    try:
        clean_expr = expression.strip().replace('x', '*').replace('X', '*')
        node = ast.parse(clean_expr, mode='eval').body
        result = eval_expr(node)
        return str(round(result, 2) if isinstance(result, float) else result)
    except Exception as e:
        return f'Calculation Error: {e}'

@tool
def get_passing_rules() -> dict:
    '''Returns the university passing rules: minimum overall average (40%) and minimum mark in each subject (35%).'''
    return {
        'minimum_overall_average_percent': 40,
        'minimum_mark_per_subject': 35,
        'rule_summary': 'Student passes if every subject mark >= 35 and overall average >= 40%.'
    }

tools = [get_student_info, get_student_marks, calculator, get_passing_rules]
print(f'Defined {len(tools)} LangChain tools:')
for t in tools:
    print(f' - {t.name}: {t.description[:60]}...')

Defined 4 LangChain tools:
 - get_student_info: Returns student name and department given their student_id....
 - get_student_marks: Returns python mark, database mark, ai mark, and web mark fo...
 - calculator: Used to calculate total marks and average marks from an arit...
 - get_passing_rules: Returns the university passing rules: minimum overall averag...


In [6]:
# Step 5: Initialize Gemini LLM (gemini-3.1-flash-lite / gemini-3.5-flash-lite) and create Tool-Calling Agent
import sys
import subprocess

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ModuleNotFoundError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "langchain-google-genai",
    ])
    from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
try:
    from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
except ImportError:
    from langchain.agents import create_tool_calling_agent, AgentExecutor

# gemini-3.1-flash-lite offers high rate limits (15 RPM) and fast tool calling
llm = ChatGoogleGenerativeAI(
    model='gemini-3.1-flash-lite',
    google_api_key=os.environ['GEMINI_API_KEY'],
    temperature=0.0,
)

system_prompt = (
    'You are a helpful university academic advisor assistant.\n'
    'Answer questions accurately by selecting and invoking the appropriate tools:\n'
    '- get_student_info(student_id): Use when asked for student name or department.\n'
    '- get_student_marks(student_id): Use when asked for subject marks (python, database, ai, web).\n'
    '- calculator(expression): ALWAYS use this for arithmetic computations (totals, averages).\n'
    '- get_passing_rules(): Use when checking passing criteria or university rules.\n\n'
    'Rules:\n'
    '1. Never guess or compute numbers in your head; invoke calculator.\n'
    '2. Autonomously select and chain tools as needed based on the user question.\n'
    '3. Provide a clear, well-structured final answer.'
)

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}'),
    MessagesPlaceholder(variable_name='agent_scratchpad'),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=10)
print('LangChain AgentExecutor successfully initialized with Gemini 3.1 Flash Lite!')

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


LangChain AgentExecutor successfully initialized with Gemini 3.1 Flash Lite!


## Testing the Agent on Required Queries

In [7]:
# Question 1: What is the name and department of student 22CS045?
# Expected Tool Flow: get_student_info()
q1 = 'What is the name and department of student 22CS045?'
res1 = agent_executor.invoke({'input': q1})
print('\n[FINAL ANSWER]:\n', res1['output'])



> Entering new AgentExecutor chain...



Invoking: `get_student_info` with `{'student_id': '22CS045'}`


{'student_id': '22CS045', 'name': 'Dhanushya', 'department': 'Computer Science'}

[{'type': 'text', 'text': 'The student with ID 22CS045 is Dhanushya, and she is in the Computer Science department.', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0TmT8yIaxCnFzIKyowGEyDG3srj0P2UE0S9YJheM8yAMK+gbHCdN08Vz0BUjz/sOGAINURUv+oreQ2f7Q4SHsR29oJnSYqYQvXjIENNpQ68Ls7t6c0yxIX91aWXSYA0H8h7WlnQE+HIJa4dA=='}}]

> Finished chain.

[FINAL ANSWER]:
 [{'type': 'text', 'text': 'The student with ID 22CS045 is Dhanushya, and she is in the Computer Science department.', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0TmT8yIaxCnFzIKyowGEyDG3srj0P2UE0S9YJheM8yAMK+gbHCdN08Vz0BUjz/sOGAINURUv+oreQ2f7Q4SHsR29oJnSYqYQvXjIENNpQ68Ls7t6c0yxIX91aWXSYA0H8h7WlnQE+HIJa4dA=='}}]


In [8]:
# Question 2: What are the marks of 22CS047?
# Expected Tool Flow: get_student_marks()
q2 = 'What are the marks of 22CS047?'
res2 = agent_executor.invoke({'input': q2})
print('\n[FINAL ANSWER]:\n', res2['output'])



> Entering new AgentExecutor chain...



Invoking: `get_student_marks` with `{'student_id': '22CS047'}`


{'student_id': '22CS047', 'python': 92, 'database': 88, 'ai': 95, 'web': 90}

[{'type': 'text', 'text': 'The marks for student 22CS047 are as follows:\n\n*   **Python:** 92\n*   **Database:** 88\n*   **AI:** 95\n*   **Web:** 90', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0TJ0NcTe7VQiLrpa+tqQxT4hQAbzNf+nt30EbUxCvnN+fZVKuMgqSThkTtw5MlzygFKUQIKqtVXeaqaMe/rVAf/ruanY6RncB4yVKGyc3+U8c9lHqyGyGnVsX3J1fWEzKaBgAIVcyMHIluyw=='}}]

> Finished chain.

[FINAL ANSWER]:
 [{'type': 'text', 'text': 'The marks for student 22CS047 are as follows:\n\n*   **Python:** 92\n*   **Database:** 88\n*   **AI:** 95\n*   **Web:** 90', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0TJ0NcTe7VQiLrpa+tqQxT4hQAbzNf+nt30EbUxCvnN+fZVKuMgqSThkTtw5MlzygFKUQIKqtVXeaqaMe/rVAf/ruanY6RncB4yVKGyc3+U8c9lHqyGyGnVsX3J1fWEzKaBgAIVcyMHIluyw=='}}]


In [9]:
# Question 3: What is the total and average mark of 22CS045?
# Expected Tool Flow: get_student_marks() -> calculator()
q3 = 'What is the total and average mark of 22CS045?'
res3 = agent_executor.invoke({'input': q3})
print('\n[FINAL ANSWER]:\n', res3['output'])



> Entering new AgentExecutor chain...



Invoking: `get_student_marks` with `{'student_id': '22CS045'}`


{'student_id': '22CS045', 'python': 85, 'database': 72, 'ai': 90, 'web': 78}


Invoking: `calculator` with `{'expression': '85 + 72 + 90 + 78'}`


325


Invoking: `calculator` with `{'expression': '325 / 4'}`


81.25

[{'type': 'text', 'text': 'For student 22CS045, the total mark is 325 and the average mark is 81.25.', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0T07lTWSsFXyeJdpGFqsWFIK46gPTJMyxbsWAA6YHWB0dJIf7LDi+WrWrkq+ou6ItseP7dSPLLbE20FEg+HC1C5zhbdZ/NbFNyfbYDzmxKQusarwukczsd/v2oOWup671HH4Wo3cZH7pBAGA=='}}]

> Finished chain.

[FINAL ANSWER]:
 [{'type': 'text', 'text': 'For student 22CS045, the total mark is 325 and the average mark is 81.25.', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0T07lTWSsFXyeJdpGFqsWFIK46gPTJMyxbsWAA6YHWB0dJIf7LDi+WrWrkq+ou6ItseP7dSPLLbE20FEg+HC1C5zhbdZ/NbFNyfbYDzmxKQusarwukczsd/v2oOWup671HH4Wo3cZH7pBAGA=='}}]


In [10]:
# Question 4: Is 22CS045 eligible to pass according to the university rules?
# Expected Tool Flow: get_student_marks() -> get_passing_rules() -> calculator()
q4 = 'Is 22CS045 eligible to pass according to the university rules?'
res4 = agent_executor.invoke({'input': q4})
print('\n[FINAL ANSWER]:\n', res4['output'])



> Entering new AgentExecutor chain...



Invoking: `get_student_marks` with `{'student_id': '22CS045'}`


{'student_id': '22CS045', 'python': 85, 'database': 72, 'ai': 90, 'web': 78}
Invoking: `get_passing_rules` with `{}`


{'minimum_overall_average_percent': 40, 'minimum_mark_per_subject': 35, 'rule_summary': 'Student passes if every subject mark >= 35 and overall average >= 40%.'}


Invoking: `calculator` with `{'expression': '(85 + 72 + 90 + 78) / 4'}`


81.25

[{'type': 'text', 'text': "To determine if student 22CS045 is eligible to pass, we compare their performance against the university's passing rules:\n\n**University Passing Rules:**\n*   **Minimum mark per subject:** 35\n*   **Minimum overall average:** 40%\n\n**Student 22CS045 Performance:**\n*   **Python:** 85\n*   **Database:** 72\n*   **AI:** 90\n*   **Web:** 78\n*   **Overall Average:** 81.25%\n\n**Evaluation:**\n1.  **Subject Marks:** All individual subject marks (85, 72, 90, 78) are well above the minimum requirement of 35.\n2.  **Overall Average:** The student's average of 81.25% is well above the minimum requirement of 40%.\n\n**Conclusion:**\nYes, student 22CS045 is eligible to pass.", 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0TaQWd9j5lTrVUapqeluEfsON7W0vEJNXnXnt0SPvyzdFfCMgCQCzL/JV+X4aTnXmAMXa26BIthVCq+rwGaPgh/QQM4q2wlGDS4ani1+oJkSa/E4EZALknNoVDM2wldQG6AsKagPsuV3gicw=='}}]

> Finished chain.

[FINAL ANSWER]:
 [{'type': 'text', 'text': "To determine if student 22CS045 i

## 🏆 Challenge Question (Multi-Tool Autonomous Chaining)

The agent should automatically determine that it needs all 4 tools sequentially:
`get_student_info()` $\rightarrow$ `get_student_marks()` $\rightarrow$ `calculator()` $\rightarrow$ `get_passing_rules()` $\rightarrow$ Final Answer

In [11]:
challenge_query = (
    'I am 22CS045. Tell me my name, department, total marks, average marks, '
    'and whether I satisfy the university passing requirements.'
)
res_challenge = agent_executor.invoke({'input': challenge_query})
print('\n[FINAL ANSWER]:\n', res_challenge['output'])



> Entering new AgentExecutor chain...



Invoking: `get_student_info` with `{'student_id': '22CS045'}`


{'student_id': '22CS045', 'name': 'Dhanushya', 'department': 'Computer Science'}
Invoking: `get_student_marks` with `{'student_id': '22CS045'}`


{'student_id': '22CS045', 'python': 85, 'database': 72, 'ai': 90, 'web': 78}
Invoking: `get_passing_rules` with `{}`


{'minimum_overall_average_percent': 40, 'minimum_mark_per_subject': 35, 'rule_summary': 'Student passes if every subject mark >= 35 and overall average >= 40%.'}


Invoking: `calculator` with `{'expression': '85 + 72 + 90 + 78'}`


325


Invoking: `calculator` with `{'expression': '325 / 4'}`


81.25

[{'type': 'text', 'text': 'Hello Dhanushya, here is your academic summary:\n\n*   **Name:** Dhanushya\n*   **Department:** Computer Science\n*   **Subject Marks:**\n    *   Python: 85\n    *   Database: 72\n    *   AI: 90\n    *   Web: 78\n*   **Total Marks:** 325\n*   **Average Marks:** 81.25%\n\n**Passing Status:**\nYes, you satisfy the university passing requirements. You have achieved an overall average of 81.25% (which is above the 40% requirement) and all your individual subject marks are well above the minimum passing mark of 35.', 'index': 0, 'extras': {'signature': 'EnEKbwFpFH0T4O56IYa0D/s41+fqCQryPMuYD9C/vusBArCTK8iD2ea2XGF8EIugkeBPD1Hg46hL98z9xHT2ilpXJ+7RAMzsr8oh0Bdohkeu1py0fIr3+sadOUnNt20gwLWaiBIk2C9zln88FsOaP9pfgA=='}}]

> Finished chain.

[FINAL ANSWER]:
 [{'type': 'text', 'text': 'Hello Dhanushya, here is your academic summary:\n\n*   **Name:** Dhanushya\n*   **Department:** Computer Science\n*   **Subject Marks:**\n    *   Python: 85\n    *   Database: 72\n    *   AI: 9